# 8× 叠加：需求、完成吞吐与积压
运行下方单元格刷新全部统计与图表。只扫描完成的8×配置，原始实验文件保持不变。

In [1]:
from pathlib import Path
from array import array
import json, re, csv, gzip, math, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPO = Path('/home/xieqiang/project/aisimulate')
ROOT = Path('/home/xieqiang/project/results/swe-chat-glm52-h200-ep8-16x')
BUNDLE = Path('/home/xieqiang/project/data/swe_chat/derived/verified_interleave_mooncake_v1/claude-code-main-glm52-128k-overlay-8x')
OUT = REPO / 'experiments/disk_swe/analysis/throughput_8x'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'cache').mkdir(exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
MANIFEST = json.loads((BUNDLE / 'manifest.json').read_text())
BIN_MS = 300000
DAY_MS = 86400000
selected = {}
for plan in sorted(ROOT.glob('*/sweep.json')):
    if plan.parent.name == '20261117T061307.092350Z':
        continue
    for case in json.loads(plan.read_text()).get('cases', []):
        if case.get('status') != 'completed' or not case['name'].endswith('_overlay_8x'):
            continue
        matches = sorted(Path(case['log']).parent.glob('*/prediction/requests.jsonl'))
        if matches:
            selected[case['name']] = dict(case, requests=str(matches[-1]), plan=str(plan))
assert selected, 'No completed 8x cases'

def number_fields(names):
    return re.compile(rb'"(' + b'|'.join(n.encode() for n in names) + rb')"\s*:\s*(-?[0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?)')

def hist(t, weights, n, start):
    indexes = np.floor((t - start) / BIN_MS).astype(np.int64)
    assert indexes.min() >= 0
    return np.bincount(indexes, weights=weights, minlength=n)[:n]

# The demand windows come from the source trace, independently of simulation outcomes.
trace_cache = OUT / 'cache/trace.npz'
trace_file = BUNDLE / 'requests.jsonl'
trace_identity = f'{trace_file}:{trace_file.stat().st_size}:{trace_file.stat().st_mtime_ns}'
identity_file = OUT / 'cache/trace_identity.txt'
if not trace_cache.exists() or not identity_file.exists() or identity_file.read_text() != trace_identity:
    cols = {k: array('d') for k in ['timestamp', 'input_length', 'output_length']}
    pat = number_fields(cols)
    with trace_file.open('rb') as stream:
        for line in stream:
            row = dict(pat.findall(line))
            for k in cols:
                cols[k].append(float(row[k.encode()]))
    np.savez_compressed(trace_cache, **{k: np.asarray(v) for k,v in cols.items()})
    identity_file.write_text(trace_identity)
    print('Source trace scanned', len(cols['timestamp']), flush=True)
trace = dict(np.load(trace_cache))
arrival = trace['timestamp']
assert len(arrival) == MANIFEST['request_count']
assert np.all(np.diff(arrival) >= 0)
start = float(arrival.min())
arrival_end = float(arrival.max()) + 1
main_end = start + MANIFEST['overlay']['segment_ms']
n = math.ceil((arrival_end - start) / BIN_MS)
edges = start + np.arange(n+1)*BIN_MS
windows = [('main', start, main_end), ('whole_arrivals', start, arrival_end), ('sparse_tail', main_end, arrival_end)]
demand_hist = {k: hist(arrival, None if k == 'requests' else trace[k], n, start)
               for k in ['requests','input_length','output_length']}
demand = []
for label, lo, hi in windows:
    mask = (arrival >= lo) & (arrival < hi)
    full_bins = (edges[:-1] >= lo) & (edges[1:] <= hi)
    secs = (hi-lo)/1000
    d = dict(window=label, start_day=(lo-start)/DAY_MS, end_day=(hi-start)/DAY_MS,
             duration_seconds=secs, requests=int(mask.sum()), request_share=float(mask.mean()),
             request_per_s=float(mask.sum()/secs),
             input_tokens=int(trace['input_length'][mask].sum()),
             expected_output_tokens=int(trace['output_length'][mask].sum()))
    d['input_token_per_s'] = d['input_tokens']/secs
    d['expected_output_token_per_s'] = d['expected_output_tokens']/secs
    for key, values in demand_hist.items():
        rates = values[full_bins]/300
        d[key+'_5min_p95_per_s'] = float(np.percentile(rates,95))
        d[key+'_5min_max_per_s'] = float(rates.max())
    demand.append(d)
assert sum(d['requests'] for d in demand if d['window'] != 'whole_arrivals') == len(arrival)
assert int(trace['input_length'].sum()) == MANIFEST['diagnostics']['emitted_input_tokens']
print('Demand windows', [(d['window'], d['requests'], round(d['request_per_s'],4)) for d in demand], flush=True)

summaries, global_rows, series = [], [], {}
names = ['arrival_time_ms','terminal_time_ms','prefill_admit_ms','input_length','output_length']
pat = number_fields(names)
decoder = json.JSONDecoder()

def read_header(path, key):
    with path.open() as stream:
        header = stream.read(20000).split('"per_request":',1)[0]
    marker = f'"{key}":'
    if marker not in header:
        return None
    return decoder.raw_decode(header.split(marker,1)[1].lstrip())[0]

for pos, (name, case) in enumerate(sorted(selected.items()),1):
    path = Path(case['requests'])
    stamp = f'{path}:{path.stat().st_size}:{path.stat().st_mtime_ns}'
    cache_path = OUT / 'cache' / (name+'.json')
    cache_npz = cache_path.with_suffix('.npz')
    cached = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    if cached.get('identity') == stamp and cache_npz.exists():
        summaries.extend(cached['windows'])
        global_rows.append(cached['global'])
        series[name] = dict(np.load(cache_npz))
        print(f'{pos}/{len(selected)} cached {name}', flush=True)
        continue
    cols = {k: array('d') for k in names}
    with path.open('rb') as stream:
        for line in stream:
            row = dict(pat.findall(line))
            assert b'"terminal_status": "completed"' in line or b'"terminal_status":"completed"' in line
            for k in cols:
                cols[k].append(float(row[k.encode()]))
    a, t, p, inp, out = [np.asarray(cols[k]) for k in names]
    # This dataset gives every row an independent session; arrival ordering is unchanged.
    assert len(a) == len(arrival)
    order = np.argsort(a, kind='stable')
    a,t,p,inp,out = [v[order] for v in [a,t,p,inp,out]]
    assert np.array_equal(a, arrival), 'Replay arrivals differ from source trace'
    # Equal timestamps may be reported in a different request order.
    verify_order=np.lexsort((out,inp,a))
    source_order=np.lexsort((trace['output_length'],trace['input_length'],arrival))
    assert np.array_equal(inp[verify_order], trace['input_length'][source_order])
    assert np.array_equal(out[verify_order], trace['output_length'][source_order])
    assert np.all(t >= a) and np.all(p >= a) and np.all(p <= t)
    # Peak in-flight requests is evaluated at every arrival, including tied arrivals.
    st, sp = np.sort(t), np.sort(p)
    peak_events = np.searchsorted(a,a,side='right') - np.searchsorted(st,a,side='right')
    peak_pwait = np.searchsorted(a,a,side='right') - np.searchsorted(sp,a,side='right')
    sample_t = edges[1:]
    inflight = np.searchsorted(a,sample_t,side='right') - np.searchsorted(st,sample_t,side='right')
    preadmit = np.searchsorted(a,sample_t,side='right') - np.searchsorted(sp,sample_t,side='right')
    # Histogram includes the arrival horizon. Draining beyond it is reported separately.
    completed_hist = hist(t, None, max(n,math.ceil((t.max()-start+1)/BIN_MS)), start)[:n]
    completed_out_hist = hist(t, out, max(n,math.ceil((t.max()-start+1)/BIN_MS)), start)[:n]
    m = re.fullmatch(r'swe_h200_ep8_p(\d+)gb_read(\d+)_overlay_8x',name)
    assert m
    dram,bw=map(int,m.groups())
    pred = path.with_name('prediction.json')
    committed = read_header(pred,'committed_prefill_tokens')
    duration_ms = read_header(pred,'duration_ms')
    assert read_header(pred,'completed_requests') == len(a)
    g = dict(case=name,dram_gb=dram,read_gbps_per_tp=bw,completed_requests=len(a),
             total_input_tokens=int(inp.sum()),total_output_tokens=int(out.sum()),
             committed_prefill_tokens=committed,duration_ms=duration_ms,
             prefill_compute_token_per_s_full_run=committed/(duration_ms/1000),
             completed_request_per_s_full_run=len(a)/(duration_ms/1000),
             output_token_per_s_full_run=out.sum()/(duration_ms/1000),
             drain_after_last_arrival_seconds=max(0,(t.max()-arrival.max())/1000),
             source_result=str(path),source_plan=case['plan'])
    rows=[]
    for label,lo,hi in windows:
        cmask=(t>=lo)&(t<hi)
        amask=(a>=lo)&(a<hi)
        secs=(hi-lo)/1000
        backlog_start=int(np.searchsorted(a,lo,side='left')-np.searchsorted(st,lo,side='left'))
        backlog_end=int(np.searchsorted(a,hi,side='left')-np.searchsorted(st,hi,side='left'))
        p_wait_end=int(np.searchsorted(a,hi,side='left')-np.searchsorted(sp,hi,side='left'))
        full_bins=(edges[:-1]>=lo)&(edges[1:]<=hi)
        row=dict(case=name,dram_gb=dram,read_gbps_per_tp=bw,window=label,
                 arrived_requests=int(amask.sum()),completed_requests=int(cmask.sum()),
                 arrival_request_per_s=amask.sum()/secs,completion_request_per_s=cmask.sum()/secs,
                 completed_output_tokens=int(out[cmask].sum()),
                 completed_output_token_per_s=out[cmask].sum()/secs,
                 backlog_start=backlog_start,backlog_end=backlog_end,backlog_delta=backlog_end-backlog_start,
                 peak_inflight_requests=int(max(backlog_start,peak_events[amask].max(initial=0))),
                 p_preadmission_waiting_end=p_wait_end,
                 peak_p_preadmission_waiting=int(max(0,peak_pwait[amask].max(initial=0))),
                 p_busy_fraction=None,d_busy_fraction=None,p_actual_compute_token_per_s=None)
        row['completion_request_5min_p95_per_s']=float(np.percentile(completed_hist[full_bins]/300,95))
        row['completion_request_5min_max_per_s']=float((completed_hist[full_bins]/300).max())
        assert row['backlog_delta'] == row['arrived_requests']-row['completed_requests']
        rows.append(row)
    ss=dict(completed_requests=completed_hist,completed_output_tokens=completed_out_hist,
            unfinished_requests=inflight,p_preadmission_waiting=preadmit)
    np.savez_compressed(cache_npz,**ss)
    cache_path.write_text(json.dumps(dict(identity=stamp,windows=rows,global_=None,**{'global':g}),ensure_ascii=False))
    summaries.extend(rows); global_rows.append(g); series[name]=ss
    print(f'{pos}/{len(selected)} analyzed {name}; main-end backlog {rows[0]["backlog_end"]}',flush=True)

def write_csv(path,rows):
    with path.open('w',newline='',encoding='utf-8-sig') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)

write_csv(OUT/'demand_summary.csv',demand)
write_csv(OUT/'configuration_summary.csv',summaries)
write_csv(OUT/'whole_run_compute.csv',global_rows)
full_main=(edges[:-1]>=start)&(edges[1:]<=main_end)
peak_index=int(np.flatnonzero(full_main)[np.argmax(demand_hist['input_length'][full_main])])
high=[]
for name,ss in series.items():
    high.append(dict(case=name,start_day=(edges[peak_index]-start)/DAY_MS,
                     end_day=(edges[peak_index+1]-start)/DAY_MS,
                     arrived_requests=int(demand_hist['requests'][peak_index]),
                     arrival_input_tokens=int(demand_hist['input_length'][peak_index]),
                     completed_requests=int(ss['completed_requests'][peak_index]),
                     completed_output_tokens=int(ss['completed_output_tokens'][peak_index]),
                     backlog_start=int(ss['unfinished_requests'][peak_index-1]) if peak_index else 0,
                     backlog_end=int(ss['unfinished_requests'][peak_index])))
write_csv(OUT/'common_peak_window.csv',high)
with gzip.open(OUT/'timeseries_5min.csv.gz','wt',newline='',encoding='utf-8') as stream:
    writer=csv.writer(stream)
    writer.writerow(['case','end_day','arrival_requests','arrival_input_tokens','expected_output_tokens',
                     'completed_requests','completed_output_tokens_attributed_at_completion',
                     'unfinished_requests','p_preadmission_waiting'])
    for name,ss in series.items():
        for i in range(n):
            writer.writerow([name,(edges[i+1]-start)/DAY_MS,*[demand_hist[k][i] for k in ['requests','input_length','output_length']],
                             *[ss[k][i] for k in ['completed_requests','completed_output_tokens','unfinished_requests','p_preadmission_waiting']]])
meta=dict(bundle=str(BUNDLE),requests_sha256=MANIFEST['files']['requests.jsonl']['sha256'],
          main_end_days=(main_end-start)/DAY_MS,whole_arrival_days=(arrival_end-start)/DAY_MS,
          window_policy='Main interval is overlay segment_ms, not first 50%; windows shared by all configurations.',
          high_peak_policy='5-minute bins anchored at first arrival; P95/max use only complete bins in each window, including zero bins.',
          output_policy='Window output throughput attributes all output tokens to request completion; actual per-token generation timing is unavailable.',
          unavailable=['P/D active execution fraction','per-window P computed tokens','actual per-window D generation tokens','sustainable throughput ceiling','GPU compute utilization'],
          configurations=len(selected),demand=demand,windows=summaries,whole_runs=global_rows,common_peak_window=high)
(OUT/'results.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2))

# Publication-style standalone charts; notebook retains all plotting code.
x=(edges[:-1]-start)/DAY_MS
plt.rcParams.update({'font.size':10,'figure.dpi':160})
fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=True)
for ax,key,label in zip(axs,['requests','input_length'],['Arriving requests / s','Raw input tokens / s']):
    ax.plot(x,demand_hist[key]/300,lw=.6,color='#14786c')
    ax.axvspan((main_end-start)/DAY_MS,(arrival_end-start)/DAY_MS,color='#eeeeee',label='Sparse tail')
    ax.axvline((main_end-start)/DAY_MS,color='#a85824',ls='--',label='Overlay target window ends')
    ax.set_ylabel(label);ax.grid(alpha=.2)
axs[0].legend(loc='upper right');axs[-1].set_xlabel('Days since first trace arrival')
fig.tight_layout();fig.savefig(OUT/'figures/demand_windows.png');plt.close(fig)

fig,axs=plt.subplots(3,1,figsize=(12,8),sharex=True)
for dram in [64,128,256,512]:
    name=f'swe_h200_ep8_p{dram}gb_read2_overlay_8x'
    if name not in series: continue
    ss=series[name]
    axs[0].plot(x,ss['completed_requests']/300,lw=.65,label=f'{dram} GB')
    axs[1].plot(x,ss['unfinished_requests'],lw=.65,label=f'{dram} GB')
    axs[2].plot(x,ss['p_preadmission_waiting'],lw=.65,label=f'{dram} GB')
for ax,label in zip(axs,['Completions / s (5-minute)','Unfinished requests','Waiting for first P admission']):
    ax.set_ylabel(label);ax.grid(alpha=.2);ax.legend(loc='upper right')
    ax.axvline((main_end-start)/DAY_MS,color='#888888',ls='--')
axs[-1].set_xlabel('Days since first trace arrival')
fig.suptitle('8x overlay, SSD read 2 GB/s per TP shard')
fig.tight_layout();fig.savefig(OUT/'figures/completion_backlog.png');plt.close(fig)

main=demand[0];whole=demand[1]
md=['# 8× 叠加：需求、完成吞吐与积压分析','',
    '## 1. 目的与统计窗口','',
    '先用已有场景 1 结果确认到达需求、实际完成和积压情况，为后续容量测试选择配置。本次没有新增仿真实验，也没有改动原有结果文档。',
    '',f'输入共 **{len(arrival):,} 条请求**；已纳入 **{len(selected)} 组完成的 8× 配置**。主要负载时段由数据包的 `overlay.segment_ms` 确定，所有配置共用，不按仿真结果或时间前 50% 划分。',
    '', '| 窗口 | 时间范围（天） | 请求数 | 占全部请求 | 平均请求/s | 输入 token/s | 预期输出 token/s |',
    '|---|---:|---:|---:|---:|---:|---:|']
labels={'main':'主要负载时段','whole_arrivals':'全到达时段','sparse_tail':'稀疏尾段'}
for d in demand:
    md.append(f'| {labels[d["window"]]} | {d["start_day"]:.2f}–{d["end_day"]:.2f} | {d["requests"]:,} | {d["request_share"]:.3%} | {d["request_per_s"]:.4f} | {d["input_token_per_s"]:,.1f} | {d["expected_output_token_per_s"]:,.1f} |')
md+=['','![到达需求与主要负载窗口](figures/demand_windows.png)','',
     '高峰用固定 5 分钟分桶统计。只使用完整落入窗口的桶，零到达桶也参与 P95；分桶起点固定为首次到达。',
     '', '| 主要负载时段的需求 | 平均 | 5 分钟速率 P95 | 5 分钟速率最大值 |','|---|---:|---:|---:|']
for label,key,avg in [('请求/s','requests','request_per_s'),('输入 token/s','input_length','input_token_per_s'),('预期输出 token/s','output_length','expected_output_token_per_s')]:
    md.append(f'| {label} | {main[avg]:,.3f} | {main[key+"_5min_p95_per_s"]:,.3f} | {main[key+"_5min_max_per_s"]:,.3f} |')
md+=['','## 2. 各配置的实际完成与积压','',
     '以下均为上述主要负载时段。完成请求按 `terminal_time_ms` 归桶；输出 token/s 是“在窗口内完成的请求，其全部输出 token 数 ÷ 窗口时长”，不能当作该窗口逐 token 实际生成速率。',
     '', '未完成请求＝已到达−已结束，覆盖 P、交接和 D，不能直接叫 P 队列。P 首次准入前等待还可能包含 KV 恢复；此处不把它全部归为计算排队。峰值未完成数按逐请求事件精确计算，图中为每 5 分钟采样。',
     '', '| P DRAM（GB） | SSD 读（GB/s/TP） | 完成请求/s | 完成归属输出 token/s | 末端未完成数 | 峰值未完成数 | 末端 P 首次准入前等待数 |',
     '|---:|---:|---:|---:|---:|---:|---:|']
for row in sorted((r for r in summaries if r['window']=='main'),key=lambda r:(r['dram_gb'],r['read_gbps_per_tp'])):
    md.append(f'| {row["dram_gb"]} | {row["read_gbps_per_tp"]} | {row["completion_request_per_s"]:.4f} | {row["completed_output_token_per_s"]:.2f} | {row["backlog_end"]:,} | {row["peak_inflight_requests"]:,} | {row["p_preadmission_waiting_end"]:,} |')
md+=['','![完成速率与未完成请求数](figures/completion_backlog.png)','',
     f'另固定输入 token 需求最高的同一个 5 分钟窗口（第 {(edges[peak_index]-start)/DAY_MS:.5f}–{(edges[peak_index+1]-start)/DAY_MS:.5f} 天）做配置对照。该窗口到达 {int(demand_hist["requests"][peak_index]):,} 条请求、{int(demand_hist["input_length"][peak_index]):,} 个输入 token；各组完成数及积压变化见 `common_peak_window.csv`。这样比较的是相同负载时段，而不是各组各自的完成高峰。','',
     '## 3. 实际 Prefill 计算量与数据边界','',
     '`committed_prefill_tokens` 来自整场报告，表示实际提交执行的 Prefill token 总量，包含重复执行。下表整场平均以报告的完整仿真时长为分母，包含稀疏尾段和排空时间；不能与主要负载时段的速率直接相除。输入 token/s 含可复用前缀，不等于 P 计算需求。',
     '', '| P DRAM（GB） | SSD 读（GB/s/TP） | 整场实际 Prefill token | 整场平均实际 Prefill token/s | 整场输出 token/s | 末次到达后排空（秒） |',
     '|---:|---:|---:|---:|---:|---:|']
for g in sorted(global_rows,key=lambda r:(r['dram_gb'],r['read_gbps_per_tp'])):
    md.append(f'| {g["dram_gb"]} | {g["read_gbps_per_tp"]} | {g["committed_prefill_tokens"]:,} | {g["prefill_compute_token_per_s_full_run"]:.2f} | {g["output_token_per_s_full_run"]:.2f} | {g["drain_after_last_arrival_seconds"]:.2f} |')
main_rows=[r for r in summaries if r['window']=='main']
same_read={r['dram_gb']:r for r in main_rows if r['read_gbps_per_tp']==2}
same_compute={r['dram_gb']:r for r in global_rows if r['read_gbps_per_tp']==2}
md+=['','这批实验关闭了调度批次轨迹记录，因此不能从现有文件准确恢复主要负载时段的 P/D 执行忙碌比例、BS、P 实际计算 token/s 和 D 实际生成 token/s。`decode_worker_seconds` 属于 worker 运行时长，不是执行忙碌时长；请求从准入到结束的时间也不能替代 GPU 执行时间。',
     '', '## 4. 当前能回答什么','',
     f'主要负载时段占 {main["end_day"]/whole["end_day"]:.1%} 的到达时间，承载 {main["request_share"]:.3%} 的请求。全程平均请求速率 {whole["request_per_s"]:.4f}/s 会被长尾稀释，主要负载时段为 {main["request_per_s"]:.4f}/s。后续容量比较应优先使用主要负载时段，并保留全程口径。',
     '',f'本次 {len(main_rows)} 组配置在主要负载窗口末均剩 {min(r["backlog_end"] for r in main_rows)}–{max(r["backlog_end"] for r in main_rows)} 条未完成请求，窗口内峰值却为 {min(r["peak_inflight_requests"] for r in main_rows)}–{max(r["peak_inflight_requests"] for r in main_rows)} 条。因此“长期大体消化”与“短时发生等待”同时存在。',
     '',f'例如固定 SSD 读 2 GB/s/TP，DRAM 256 GB、128 GB、64 GB 的峰值未完成请求数分别为 {same_read[256]["peak_inflight_requests"]}、{same_read[128]["peak_inflight_requests"]}、{same_read[64]["peak_inflight_requests"]}。64 GB 组的整场实际 Prefill 计算 token 比 256 GB 组多 {same_compute[64]["committed_prefill_tokens"]/same_compute[256]["committed_prefill_tokens"]-1:.2%}。缩容不只改变传输，还改变了实际计算工作量；不能直接把三组当成相同计算需求。',
     '', '完成速率接近到达速率且窗口末积压小，说明在这个时间尺度上系统大体消化了需求，但不代表没有短时拥塞；还应看峰值积压和高峰分桶。积压并不能单独证明 GPU 算力饱和，KV 恢复、HBM 容量和调度限制都可能造成等待。',
     '', '本分析尚未测得最大可持续吞吐，因此不能给出“算力被吃满百分之多少”。下一步可根据积压和完成曲线选代表配置，补执行轨迹及同负载形状的容量测试。',
     '', '## 5. 文件与复现','',
     '- `analysis.ipynb`：全部数据读取、校验、统计和绘图代码；重跑会刷新结果，未变的逐请求文件使用缓存。',
     '- `demand_summary.csv`：三个输入统计窗口与需求高峰。',
     '- `configuration_summary.csv`：各配置三个窗口的完成吞吐、精确积压和高峰。',
     '- `whole_run_compute.csv`：整场实际计算量、整场吞吐及结果来源。',
     '- `common_peak_window.csv`：输入 token 需求最高的同一 5 分钟，各组完成与积压对照。',
     '- `timeseries_5min.csv.gz`：到达、完成、未完成数与首次准入前等待数的分时数据。',
     '- `results.json`：配置来源、口径和结构化结果。',
     '- `figures/`：可独立导出的图；`cache/`：避免重复扫描大文件的派生缓存。',
     '', '校验：各配置请求数、到达时间、输入和输出长度与输入 trace 一致；三个窗口的积压守恒；报告完成数与逐请求完成数一致。仅流式扫描 JSONL 和读取报告头部，不把大报告整体载入内存。','']
(OUT/'analysis.md').write_text('\n'.join(md),encoding='utf-8')
print('Saved analysis:',OUT,flush=True)


Demand windows [('main', 603973, 0.2413), ('whole_arrivals', 607190, 0.1476), ('sparse_tail', 3217, 0.002)]
1/30 cached swe_h200_ep8_p1000gb_read16_overlay_8x
2/30 cached swe_h200_ep8_p1000gb_read1_overlay_8x
3/30 cached swe_h200_ep8_p1000gb_read2_overlay_8x
4/30 cached swe_h200_ep8_p1000gb_read4_overlay_8x
5/30 cached swe_h200_ep8_p1000gb_read8_overlay_8x
6/30 cached swe_h200_ep8_p128gb_read16_overlay_8x
7/30 cached swe_h200_ep8_p128gb_read1_overlay_8x
8/30 cached swe_h200_ep8_p128gb_read2_overlay_8x
9/30 cached swe_h200_ep8_p128gb_read4_overlay_8x
10/30 cached swe_h200_ep8_p128gb_read8_overlay_8x
11/30 cached swe_h200_ep8_p256gb_read16_overlay_8x
12/30 cached swe_h200_ep8_p256gb_read1_overlay_8x
13/30 cached swe_h200_ep8_p256gb_read2_overlay_8x
14/30 cached swe_h200_ep8_p256gb_read4_overlay_8x
15/30 cached swe_h200_ep8_p256gb_read8_overlay_8x
16/30 cached swe_h200_ep8_p384gb_read16_overlay_8x
17/30 cached swe_h200_ep8_p384gb_read1_overlay_8x
18/30 cached swe_h200_ep8_p384gb_read2_ove

## 三个统计窗口的需求与完成吞吐
运行本单元格用已有缓存生成三张表，无需重扫逐请求结果。运行全部单元格也会保留此格式。

In [2]:
from pathlib import Path
import json, csv, re
import numpy as np

folder = Path('/home/xieqiang/project/aisimulate/experiments/disk_swe/analysis/throughput_8x')
report = json.loads((folder / 'results.json').read_text())
demands = {d['window']: d for d in report['demand']}
peak_rows = report['common_peak_window']
trace = np.load(folder / 'cache/trace.npz')
first_ms = float(trace['timestamp'].min())
lo = first_ms + peak_rows[0]['start_day'] * 86400000
hi = first_ms + peak_rows[0]['end_day'] * 86400000
# Round recovered boundaries to integer milliseconds to avoid float boundary drift.
lo, hi = round(lo), round(hi)
mask = (trace['timestamp'] >= lo) & (trace['timestamp'] < hi)
peak_expected = int(trace['output_length'][mask].sum())
peak_seconds = (hi - lo) / 1000
assert peak_seconds == 300
assert int(mask.sum()) == peak_rows[0]['arrived_requests']
config = {g['case']: g for g in report['whole_runs']}
rows = []
for p in peak_rows:
    g = config[p['case']]
    assert p['backlog_end'] - p['backlog_start'] == p['arrived_requests'] - p['completed_requests']
    rows.append(dict(window='peak_5min', case=p['case'], dram_gb=g['dram_gb'],
        read_gbps_per_tp=g['read_gbps_per_tp'], duration_seconds=peak_seconds,
        arrived_requests=p['arrived_requests'], completed_requests=p['completed_requests'],
        arrival_request_per_s=p['arrived_requests']/peak_seconds,
        arrival_input_token_per_s=p['arrival_input_tokens']/peak_seconds,
        completion_request_per_s=p['completed_requests']/peak_seconds,
        expected_output_token_per_s=peak_expected/peak_seconds,
        completed_output_token_per_s=p['completed_output_tokens']/peak_seconds,
        backlog_start=p['backlog_start'], backlog_end=p['backlog_end'],
        backlog_delta=p['backlog_end']-p['backlog_start'], peak_inflight_requests=None))
for w in ['main', 'whole_arrivals']:
    for r in report['windows']:
        if r['window'] != w:
            continue
        d = demands[w]
        rows.append(dict(window=w, case=r['case'], dram_gb=r['dram_gb'],
            read_gbps_per_tp=r['read_gbps_per_tp'], duration_seconds=d['duration_seconds'],
            arrived_requests=r['arrived_requests'], completed_requests=r['completed_requests'],
            arrival_request_per_s=r['arrival_request_per_s'],
            arrival_input_token_per_s=d['input_token_per_s'],
            completion_request_per_s=r['completion_request_per_s'],
            expected_output_token_per_s=d['expected_output_token_per_s'],
            completed_output_token_per_s=r['completed_output_token_per_s'],
            backlog_start=r['backlog_start'], backlog_end=r['backlog_end'],
            backlog_delta=r['backlog_delta'], peak_inflight_requests=r['peak_inflight_requests']))
assert len(rows) == 3 * report['configurations']
for r in rows:
    assert r['backlog_delta'] == r['arrived_requests'] - r['completed_requests']
with (folder / 'throughput_windows.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
report['throughput_windows'] = rows
report['peak_expected_output_tokens'] = peak_expected
(folder / 'results.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))

lines = ['## 2. 三个时段的到达需求与实际完成吞吐', '',
    '以下三表均覆盖同一批 30 组配置，使用相同输入确定时间窗口。完成请求按 terminal_time_ms 计入；输出吞吐按窗口内完成请求的全部输出 token 归属统计，不是逐 token 实际生成速率。全部时段截止于最后一次到达，之后的排空耗时见第 3 节。', '',
    '到达输入 token/s 为窗口内到达请求的完整输入 token 总量除以窗口时长，包含可复用前缀，不等于 P 实际计算 token/s。现有结果不能恢复分时段的实际计算速率，整场实际计算量见第 3 节。', '',
    '未完成请求覆盖 P、交接与 D；其数量不能直接当成 P 的计算队列。高峰表给出窗口初末积压；主要负载与全部时段额外列出逐请求事件计算的精确峰值积压。', '']
windows = [
    ('peak_5min', '2.1 高峰时段', f'固定输入 token 总量最高的同一个 5 分钟：第 {(lo-first_ms)/86400000:.5f}–{(hi-first_ms)/86400000:.5f} 天。该窗口到达 {int(mask.sum()):,} 条请求，{peak_rows[0]["arrival_input_tokens"]:,} 个输入 token，预期输出 {peak_expected:,} 个 token。最高输入量窗口与最高请求数窗口不一定相同，此处各配置使用同一个窗口。'),
    ('main', '2.2 主要负载时段', f'前 {demands["main"]["end_day"]:.5f} 天，共 {demands["main"]["requests"]:,} 条到达请求，占全部请求 {demands["main"]["request_share"]:.3%}。'),
    ('whole_arrivals', '2.3 全部时段', f'首次到达至最后一次到达，约 {demands["whole_arrivals"]["end_day"]:.5f} 天，共 {demands["whole_arrivals"]["requests"]:,} 条请求，包含稀疏长尾。各组使用同一时长作为分母。')]
for w, title, note in windows:
    lines += ['### ' + title, '', note, '']
    header = '| P DRAM（GB） | SSD 读（GB/s/TP） | 到达请求/s | 完成请求/s | 到达输入 token/s | 预期输出 token/s | 完成归属输出 token/s | 窗口初积压 | 窗口末积压 |'
    align = '|---:|---:|---:|---:|---:|---:|---:|---:|---:|'
    if w != 'peak_5min':
        header += ' 峰值积压 |'
        align += '---:|'
    lines += [header, align]
    for r in sorted((r for r in rows if r['window'] == w),
                    key=lambda r: (r['dram_gb'], r['read_gbps_per_tp'])):
        row = f'| {r["dram_gb"]} | {r["read_gbps_per_tp"]} | {r["arrival_request_per_s"]:.4f} | {r["completion_request_per_s"]:.4f} | {r["arrival_input_token_per_s"]:,.2f} | {r["expected_output_token_per_s"]:.2f} | {r["completed_output_token_per_s"]:.2f} | {r["backlog_start"]:,} | {r["backlog_end"]:,} |'
        if w != 'peak_5min':
            row += f' {r["peak_inflight_requests"]:,} |'
        lines.append(row)
    lines.append('')
lines += ['![完成速率与未完成请求数](figures/completion_backlog.png)', '',
          '上述三表的到达、完成数量及完整精度见 throughput_windows.csv。高峰的请求数差应结合窗口初末积压一起看，窗口内完成的请求可能来自窗口前。', '']
path = folder / 'analysis.md'
md = path.read_text()
prefix, tail = md.split('## 2.', 1)
_, suffix = tail.split('## 3.', 1)
md = prefix + '\n'.join(lines) + '\n## 3.' + suffix
md = md.replace('均剩 3–3 条未完成请求', '均剩 3 条未完成请求')
entry = '- `throughput_windows.csv`：高峰、主要负载和全部时段的统一需求与完成吞吐表（含数量与完整精度）。\n'
marker = '- `configuration_summary.csv`：'
if entry not in md:
    md = md.replace(marker, entry + marker)
html_link='\n交互展示：[GLM 吞吐与等待 HTML](../../throughput_glm52.html)，可选择单个配置、查看五分钟曲线并导出数据。\n'
if html_link not in md:
    md=md.replace('## 1. 目的与统计窗口',html_link+'\n## 1. 目的与统计窗口')
path.write_text(md, encoding='utf-8')
print(f'Updated three windows: {len(rows)} rows; peak expected output {peak_expected/peak_seconds:.3f} token/s')


Updated three windows: 90 rows; peak expected output 141.103 token/s
